In [2]:

"""  TASK 1: Web Scraping
● Use Python libraries like BeautifulSoup or Scrapy to extract data from websites.
● Identify and collect relevant datasets from public web pages.
● If you don’t code, use automated tools like Octoparse or ParseHub.
● Learn to handle HTML structure and web navigation to gather accurate data.
● Create custom datasets tailored to specific analysis need"""
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://books.toscrape.com/catalogue/"
START_URL = "https://books.toscrape.com/catalogue/page-1.html"

# Mapping string star ratings (One, Two, Three...) to numerical values
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}


def get_book_links(page_url):
    """Collects detail page links of all books from a catalogue page."""
    resp = requests.get(page_url, headers=HEADERS)
    if resp.status_code != 200:
        return [], None

    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    for article in soup.find_all("article", class_="product_pod"):
        href = article.h3.a["href"]
        links.append(BASE_URL + href)

    # Return next page URL if it exists
    next_btn = soup.find("li", class_="next")
    next_url = None
    if next_btn:
        next_url = BASE_URL + next_btn.a["href"]

    return links, next_url


def scrape_book_detail(url):
    """Extracts title, price, rating, availability, and description from a book detail page."""
    resp = requests.get(url, headers=HEADERS)
    if resp.status_code != 200:
        return None

    soup = BeautifulSoup(resp.text, "html.parser")

    title = soup.find("div", class_="product_main").h1.text.strip()

    price_text = soup.find("p", class_="price_color").text.strip()
    price = float(price_text.replace("£", "").replace("Â", ""))

    rating_tag = soup.find("p", class_="star-rating")
    rating_word = rating_tag["class"][1]  # e.g., "Three"
    rating = RATING_MAP.get(rating_word, None)

    availability = soup.find("p", class_="instock availability").text.strip()

    desc_tag = soup.find("div", id="product_description")
    if desc_tag:
        description = desc_tag.find_next_sibling("p").text.strip()
    else:
        description = ""

    return {
        "title": title,
        "price_gbp": price,
        "rating": rating,
        "availability": availability,
        "description": description,
        "url": url,
    }


def main(max_pages=5):
    all_books = []
    page_url = START_URL
    page_num = 1

    while page_url and page_num <= max_pages:
        print(f"Scraping page {page_num}: {page_url}")
        links, next_url = get_book_links(page_url)

        for link in links:
            book_data = scrape_book_detail(link)
            if book_data:
                all_books.append(book_data)
            time.sleep(0.3)  # Delay to avoid overloading the server

        page_url = next_url
        page_num += 1

    df = pd.DataFrame(all_books)

    # Basic cleaning: drop rows with missing essential values
    df = df.dropna(subset=["title", "price_gbp", "rating"])

    df.to_csv("data.csv", index=False, encoding="utf-8-sig")
    print(f"\nCompleted! {len(df)} books saved to data.csv.")
    print(df.head())


if __name__ == "__main__":
    # max_pages=5 -> collects approximately 100 books. Increase to collect more (50 pages in total).
    main(max_pages=5)

Scraping page 1: https://books.toscrape.com/catalogue/page-1.html
Scraping page 2: https://books.toscrape.com/catalogue/page-2.html
Scraping page 3: https://books.toscrape.com/catalogue/page-3.html
Scraping page 4: https://books.toscrape.com/catalogue/page-4.html
Scraping page 5: https://books.toscrape.com/catalogue/page-5.html

Completed! 100 books saved to data.csv.
                                   title  price_gbp  rating  \
0                   A Light in the Attic      51.77       3   
1                     Tipping the Velvet      53.74       1   
2                             Soumission      50.10       1   
3                          Sharp Objects      47.82       4   
4  Sapiens: A Brief History of Humankind      54.23       5   

              availability                                        description  \
0  In stock (22 available)  It's hard to imagine a world without A Light i...   
1  In stock (20 available)  "Erotic and absorbing...Written with starling ...   
2  In s

"My code sends a request to the book website (requests), reads the returned HTML (BeautifulSoup), extracts the title/price/rating/description from each book, then follows the 'Next' button across 5 pages, collects everything into a table (pandas), and saves it as a CSV file."